<a href="https://colab.research.google.com/github/afullhart/SantaRita/blob/main/GEE/Export_Maps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%reset -f

In [2]:
import ee
import math
import time

ee.Authenticate()
geeusername = 'andrewfullhart' #Enter your GEE user name.
geeproject = 'ee-andrewfullhart' #Enter your GEE project name.
ee.Initialize(project=geeproject)


In [ ]:

# =========================================================================
# SETUP & ASSETS (SENTINEL-2 SPECIFIC)
# =========================================================================
fc = ee.FeatureCollection('projects/ee-andrewfullhart/assets/SR_s2_model_grid_utm')
bounds_fc = ee.FeatureCollection('projects/ee-andrewfullhart/assets/SR_bounds')
cloud_windows = ee.FeatureCollection('projects/ee-andrewfullhart/assets/Cloud_FeatureClass_S2')
bounds_geom = bounds_fc.first().geometry().bounds()

dem = ee.Image('projects/ee-andrewfullhart/assets/SR_10m_DEM_Resampled')
terrain_slope = ee.Terrain.slope(dem).multiply(math.pi / 180)
terrain_aspect = ee.Terrain.aspect(dem).multiply(math.pi / 180)

# =========================================================================
# MODEL TRAINING
# =========================================================================
print("Setting up Sentinel-2 Gradient Tree Boost models...")

# S2 specific predictors (includes MCARI)
inputProps = ['B2', 'B3', 'B4', 'B5', 'B8', 'B11', 'B12', 'NDVI', 'MCARI', 'BSI', 'NBR2', 'slope', 'illumination', 'aspect']

# Core Dataset
core_training_fc = fc.filter(ee.Filter.notNull(inputProps + ['Herb_pct', 'Woody_pct']))

# HWR Dataset (Log Transformed)
def add_log_hwr(ft):
    raw_hwr = ee.Number(ft.get('Herb_Woody_Ratio'))
    log_hwr = raw_hwr.add(1).log()
    return ft.set('Log_HWR', log_hwr)

hwr_training_fc = core_training_fc.filter(ee.Filter.notNull(['Herb_Woody_Ratio'])).map(add_log_hwr)

hyperpars = {
    'numberOfTrees': 400,
    'shrinkage': 0.05,
    'samplingRate': 0.7,
    'maxNodes': 32,
    'loss': 'Huber',
    'seed': 123
}

model_bgr = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(core_training_fc, 'BGR', inputProps)
model_lpi = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(core_training_fc, 'LPI', inputProps)
model_mft = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(core_training_fc, 'MFT', inputProps)
model_herb = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(core_training_fc, 'Herb_pct', inputProps)
model_woody = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(core_training_fc, 'Woody_pct', inputProps)
model_hwr = ee.Classifier.smileGradientTreeBoost(**hyperpars).setOutputMode('REGRESSION').train(hwr_training_fc, 'Log_HWR', inputProps)

# =========================================================================
# SENTINEL-2 EXTRACTION PIPELINE
# =========================================================================
projSent2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(bounds_geom).first().select('B2').projection()

def buildS2Composite(startDate, endDate):
    sent2_ic = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(bounds_geom) \
        .filterDate(startDate, endDate)

    # Filter out images missing the MSK_CLDPRB band
    def flag_aux(img):
        return img.set('has_aux_bands', img.bandNames().contains('MSK_CLDPRB'))

    sent2_ic = sent2_ic.map(flag_aux).filter(ee.Filter.eq('has_aux_bands', True))

    def process_img(img):
        probMask = img.select('MSK_CLDPRB').lt(20)
        scl = img.select('SCL')
        # Use .And() for Earth Engine server-side boolean logic
        sclMask = scl.neq(8).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)).And(scl.neq(3))
        blueMask = img.select('B2').lt(2500)

        masterMask = probMask.And(sclMask).And(blueMask)
        maskedImg = img.updateMask(masterMask)

        sz_num = ee.Number(img.get('MEAN_SOLAR_ZENITH_ANGLE')).multiply(math.pi / 180)
        sa_num = ee.Number(img.get('MEAN_SOLAR_AZIMUTH_ANGLE')).multiply(math.pi / 180)

        cosZ = ee.Image.constant(sz_num.cos())
        sinZ = ee.Image.constant(sz_num.sin())
        sa_img = ee.Image.constant(sa_num)

        cosS = terrain_slope.cos()
        sinS = terrain_slope.sin()
        cosAzAsp = sa_img.subtract(terrain_aspect).cos()

        illumination = cosZ.multiply(cosS).add(sinZ.multiply(sinS).multiply(cosAzAsp)).rename('illumination')

        return maskedImg.addBands(illumination)

    sent2_ic = sent2_ic.map(process_img)
    sent2_median = sent2_ic.median().clip(bounds_geom)

    optical_bands = sent2_median.select(['B2', 'B3', 'B4', 'B5', 'B8', 'B11', 'B12']) \
        .multiply(0.0001) \
        .setDefaultProjection(crs=projSent2.crs(), scale=projSent2.nominalScale())

    ndvi = optical_bands.normalizedDifference(['B8', 'B4']).rename('NDVI')

    mcari = optical_bands.expression('((B5 - B4) - 0.2 * (B5 - B3)) * (B5 / B4)', {
        'B3': optical_bands.select('B3'),
        'B4': optical_bands.select('B4'),
        'B5': optical_bands.select('B5')
    }).rename('MCARI')

    bsi = optical_bands.expression('((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))', {
        'B2': optical_bands.select('B2'),
        'B4': optical_bands.select('B4'),
        'B8': optical_bands.select('B8'),
        'B11': optical_bands.select('B11')
    }).rename('BSI')

    nbr2 = optical_bands.normalizedDifference(['B11', 'B12']).rename('NBR2')

    return optical_bands.addBands([
        ndvi, mcari, bsi, nbr2,
        sent2_median.select('illumination'),
        terrain_slope.rename('slope'),
        terrain_aspect.rename('aspect')
    ])

# =========================================================================
# AUTOMATED TASK SUBMISSION
# =========================================================================
print("Fetching valid S2 cloud windows from asset (this takes a moment)...")
# Pull the actual data dictionary of the FeatureCollection from the server to Python
windows_list = cloud_windows.getInfo().get('features', [])

print(f"Found {len(windows_list)} valid Sentinel-2 windows. Submitting export tasks...")

for window in windows_list:
    props = window['properties']
    year = int(props['Year'])
    month = int(props['Month'])
    start_date_str = props['Start_Date']

    print(f"Queueing task for {year}-{month:02d}...")

    # Sentinel-2 uses a strict 7-day window advancing from the start date
    ee_start_date = ee.String(start_date_str)
    ee_end_date = ee.Date(ee_start_date).advance(7, 'day')

    s2_img = buildS2Composite(ee_start_date, ee_end_date)

    p_bgr = s2_img.classify(model_bgr).rename('Pred_BGR')
    p_lpi = s2_img.classify(model_lpi).rename('Pred_LPI')
    p_mft = s2_img.classify(model_mft).rename('Pred_MFT')
    p_herb = s2_img.classify(model_herb).rename('Pred_Herb_pct')
    p_woody = s2_img.classify(model_woody).rename('Pred_Woody_pct')
    p_hwr = s2_img.classify(model_hwr).rename('Pred_Log_HWR')

    # Stack all 6 prediction bands into a single image
    final_pred_img = ee.Image.cat([p_bgr, p_lpi, p_mft, p_herb, p_woody, p_hwr]).toFloat()

    task_name = f'SRER_S2_Predictive_Maps_{year}_{month:02d}'

    # Configure the Google Drive export task
    task = ee.batch.Export.image.toDrive(
        image=final_pred_img,
        description=task_name,
        folder='SRER_S2_Predictive_Maps',
        fileNamePrefix=task_name,
        region=bounds_geom,
        scale=10,            # Sentinel-2 native 10m resolution
        crs='EPSG:32612',    # UTM Zone 12N
        maxPixels=1e13
    )

    # Start the task on Google's servers
    task.start()
    time.sleep(0.2) # Small delay to prevent hitting the API request rate limit

print("All tasks have been successfully submitted!")
print("You can monitor their progress at: https://code.earthengine.google.com/tasks")